# LeetCode 1254 — Number of Closed Islands bằng Python

Link đề: https://leetcode.com/problems/number-of-closed-islands/ · Độ khó: **Medium** · Chủ đề: Array, Depth-First Search, Breadth-First Search, Union-Find, Matrix

Skeleton LeetCode:

```python
class Solution:
    def closedIsland(self, grid: list[list[int]]) -> int:
        ...
```

Ghi chú: ở bài này **`0` là đất, `1` là nước** — ngược với bài 200 (Number of Islands). Đây là chỗ người mới hay nhầm nhất.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Mô hình hoá lưới 2 chiều như một **đồ thị ẩn**: mỗi ô là một đỉnh, nối với tối đa 4 ô kề (lên, xuống, trái, phải).
2. Nắm chắc ba công cụ duyệt **thành phần liên thông** trên lưới: **DFS đệ quy**, **BFS bằng `deque`**, và **Union-Find (DSU)**.
3. Biến định nghĩa "đảo đóng = bị nước bao kín bốn phía" thành điều kiện cực đơn giản: **đảo đóng ⇔ không có ô nào nằm trên biên lưới**.
4. Biết hai chiến lược kinh điển: (a) **lấp biên trước** rồi đếm đảo; (b) **DFS trả về cờ `closed`** và kết hợp bằng `and` mà không bị "đoản mạch".
5. Tránh các lỗi: dừng duyệt sớm khi chạm biên, dùng `and` làm mất nhánh, nhầm `0`/`1`, tràn đệ quy, sửa luôn input.

## 2. Đề bài & ví dụ

Cho lưới 2D `grid` gồm các số `0` (**đất**) và `1` (**nước**).

- Một **đảo** là một nhóm **cực đại** các ô `0` liên thông theo **4 hướng** (trên, dưới, trái, phải — không tính đường chéo).
- Một **đảo đóng** là đảo được bao **hoàn toàn** bởi nước ở mọi phía (trái, trên, phải, dưới).

Trả về **số đảo đóng**.

**Ví dụ 1:**
```
grid = [[1,1,1,1,1,1,1,0],
        [1,0,0,0,0,1,1,0],
        [1,0,1,0,1,1,1,0],
        [1,0,0,0,0,1,0,1],
        [1,1,1,1,1,1,1,0]]
```
Kết quả: `2`. Có 4 đảo; đảo lớn ở hàng 1–3, cột 1–4 là đóng; ô `0` đơn lẻ ở (3, 6) cũng đóng; hai đảo nằm ở cột 7 (cột biên) thì **không** đóng.

**Ví dụ 2:** `grid = [[0,0,1,0,0],[0,1,0,1,0],[0,1,1,1,0]]` → `1` (chỉ ô (1, 2) là đảo đóng).

**Ví dụ 3:** lưới 7×7 với một "vòng" đất bao quanh một vòng nước, bên trong có một ô đất → `2` (đảo ngoài và đảo trong đều đóng).

**Ràng buộc:**
- `1 <= grid.length, grid[0].length <= 100`
- `0 <= grid[i][j] <= 1`

In [1]:
import random
import sys
import time
from collections import deque
from typing import List

EX1 = [[1,1,1,1,1,1,1,0],
       [1,0,0,0,0,1,1,0],
       [1,0,1,0,1,1,1,0],
       [1,0,0,0,0,1,0,1],
       [1,1,1,1,1,1,1,0]]
EX2 = [[0,0,1,0,0],
       [0,1,0,1,0],
       [0,1,1,1,0]]
EX3 = [[1,1,1,1,1,1,1],
       [1,0,0,0,0,0,1],
       [1,0,1,1,1,0,1],
       [1,0,1,0,1,0,1],
       [1,0,1,1,1,0,1],
       [1,0,0,0,0,0,1],
       [1,1,1,1,1,1,1]]
examples = [(EX1, 2), (EX2, 1), (EX3, 2)]

def show(grid):
    """In lưới: '.' là đất (0), '#' là nước (1) cho dễ nhìn."""
    for row in grid:
        print(' '.join('.' if v == 0 else '#' for v in row))

for g, want in examples:
    show(g)
    print('-> mong đợi', want, '\n')

# # # # # # # .
# . . . . # # .
# . # . # # # .
# . . . . # . #
# # # # # # # .
-> mong đợi 2 

. . # . .
. # . # .
. # # # .
-> mong đợi 1 

# # # # # # #
# . . . . . #
# . # # # . #
# . # . # . #
# . # # # . #
# . . . . . #
# # # # # # #
-> mong đợi 2 



## 3. Kiến thức nền

### 3.1 Lưới = list of lists

Trong Python, lưới `m × n` là một danh sách gồm `m` hàng, mỗi hàng là danh sách `n` số. Truy cập ô hàng `r`, cột `c` bằng `grid[r][c]`. Số hàng là `len(grid)`, số cột là `len(grid[0])`.

Một ô `(r, c)` nằm **trên biên** khi `r == 0` hoặc `r == m-1` hoặc `c == 0` hoặc `c == n-1`.

In [2]:
g = EX2
m, n = len(g), len(g[0])
print('m =', m, 'n =', n)
print('g[1][2] =', g[1][2])

def on_border(r, c, m, n):
    return r == 0 or r == m - 1 or c == 0 or c == n - 1

print([(r, c) for r in range(m) for c in range(n) if on_border(r, c, m, n)][:6], '...')
print('(1,2) trên biên?', on_border(1, 2, m, n))

m = 3 n = 5
g[1][2] = 0
[(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 0)] ...
(1,2) trên biên? False


### 3.2 Bốn hướng di chuyển

Thay vì viết 4 lệnh `if` lặp lại, ta dùng một danh sách độ dời `DIRS = [(1,0),(-1,0),(0,1),(0,-1)]`. Với mỗi `(dr, dc)` ta có ô kề `(r+dr, c+dc)`; nhớ **kiểm tra còn trong lưới** (`0 <= nr < m and 0 <= nc < n`) trước khi đọc `grid[nr][nc]`.

Lưu ý: chỉ số âm trong Python **không gây lỗi** mà lấy phần tử từ cuối (`grid[-1]` là hàng cuối) — vì vậy quên kiểm tra biên sẽ cho kết quả sai một cách "im lặng".

In [3]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def neighbors(r, c, m, n):
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n:
            yield nr, nc

print('Kề của (0,0) trong lưới 3x5:', list(neighbors(0, 0, 3, 5)))
print('Kề của (1,2) trong lưới 3x5:', list(neighbors(1, 2, 3, 5)))
print('Cạm bẫy chỉ số âm: EX2[-1][0] =', EX2[-1][0], '(không lỗi, nhưng là hàng cuối!)')

Kề của (0,0) trong lưới 3x5: [(1, 0), (0, 1)]
Kề của (1,2) trong lưới 3x5: [(2, 2), (0, 2), (1, 3), (1, 1)]
Cạm bẫy chỉ số âm: EX2[-1][0] = 0 (không lỗi, nhưng là hàng cuối!)


### 3.3 DFS đệ quy

DFS (Depth-First Search — tìm kiếm theo chiều sâu): từ một ô đất, đánh dấu đã thăm rồi **gọi đệ quy** sang từng ô kề còn là đất chưa thăm. Khi hàm kết thúc, toàn bộ thành phần liên thông đã được đánh dấu.

Python giới hạn độ sâu đệ quy mặc định ~1000. Lưới 100×100 có thể có một đảo hình "con rắn" dài ~5000 ô → cần `sys.setrecursionlimit` lớn hơn, hoặc chuyển sang **stack tường minh**/BFS.

In [4]:
def dfs_mark(grid, r, c, seen):
    m, n = len(grid), len(grid[0])
    seen.add((r, c))
    for nr, nc in neighbors(r, c, m, n):
        if grid[nr][nc] == 0 and (nr, nc) not in seen:
            dfs_mark(grid, nr, nc, seen)

seen = set()
dfs_mark(EX2, 1, 2, seen)
print('Thành phần chứa (1,2):', sorted(seen))
seen = set()
dfs_mark(EX2, 0, 0, seen)
print('Thành phần chứa (0,0):', sorted(seen))
print('Giới hạn đệ quy hiện tại:', sys.getrecursionlimit())

Thành phần chứa (1,2): [(1, 2)]
Thành phần chứa (0,0): [(0, 0), (0, 1), (1, 0), (2, 0)]
Giới hạn đệ quy hiện tại: 1000


### 3.4 BFS với `collections.deque`

BFS (Breadth-First Search — theo chiều rộng) dùng **hàng đợi**: lấy ô ở đầu hàng (`popleft`, O(1)), đẩy các ô kề hợp lệ vào cuối (`append`). Quan trọng: **đánh dấu đã thăm ngay khi đẩy vào hàng đợi**, nếu không một ô có thể bị đẩy nhiều lần.

BFS không dùng đệ quy nên **không lo tràn stack**.

In [5]:
def bfs_component(grid, sr, sc):
    m, n = len(grid), len(grid[0])
    seen = {(sr, sc)}
    q = deque([(sr, sc)])
    order = []
    while q:
        r, c = q.popleft()
        order.append((r, c))
        for nr, nc in neighbors(r, c, m, n):
            if grid[nr][nc] == 0 and (nr, nc) not in seen:
                seen.add((nr, nc))      # đánh dấu khi ĐẨY vào
                q.append((nr, nc))
    return order

print('Thứ tự BFS từ (0,0) trong EX2:', bfs_component(EX2, 0, 0))

Thứ tự BFS từ (0,0) trong EX2: [(0, 0), (1, 0), (0, 1), (2, 0)]


### 3.5 Sao chép lưới trước khi sửa

Nhiều lời giải "đánh dấu" bằng cách ghi đè `grid[r][c] = 1`. Như vậy là **sửa input** của người gọi. Muốn an toàn, sao chép trước: `g = [row[:] for row in grid]`.

Cẩn thận: `grid[:]` hay `list(grid)` chỉ là **bản sao nông** — các hàng bên trong vẫn dùng chung!

In [6]:
a = [[0, 0], [0, 0]]
shallow = a[:]
deep = [row[:] for row in a]
shallow[0][0] = 9
print('a sau khi sửa shallow:', a, '<- bị ảnh hưởng!')
deep[1][1] = 7
print('a sau khi sửa deep   :', a, '<- không đổi')

a sau khi sửa shallow: [[9, 0], [0, 0]] <- bị ảnh hưởng!
a sau khi sửa deep   : [[9, 0], [0, 0]] <- không đổi


### 3.6 Union-Find (Disjoint Set Union) tối giản

Union-Find quản lý các **nhóm rời nhau**: `find(x)` trả về "đại diện" của nhóm chứa `x`; `union(a, b)` gộp hai nhóm. Dùng **nén đường đi** (path compression) để gần như O(1) mỗi thao tác.

Ta sẽ dùng nó ở Cách 5: gộp các ô đất kề nhau, và gộp mọi ô đất ở biên vào một **đỉnh ảo "BIÊN"**.

In [7]:
parent = list(range(6))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]   # nén đường đi (halving)
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb

union(0, 1); union(1, 2); union(4, 5)
print('Nhóm của 0,1,2:', find(0), find(1), find(2))
print('Nhóm của 3:', find(3), '| Nhóm của 4,5:', find(4), find(5))
print('Số nhóm:', len({find(x) for x in range(6)}))

Nhóm của 0,1,2: 2 2 2
Nhóm của 3: 3 | Nhóm của 4,5: 5 5
Số nhóm: 3


## 4. Ý tưởng nền tảng / trực giác

Tính bằng tay với **Ví dụ 2**:

```
. . # . .        (0,0)(0,1) # (0,3)(0,4)
. # . # .        (1,0)  #  (1,2)  #  (1,4)
. # # # .        (2,0)  #    #    #  (2,4)
```

1. Đảo A = {(0,0),(0,1),(1,0),(2,0)} — có ô (0,0) nằm ở biên ⇒ có một phía "nhìn ra ngoài lưới", không bị nước chặn ⇒ **không đóng**.
2. Đảo B = {(0,3),(0,4),(1,4),(2,4)} — chạm biên ⇒ **không đóng**.
3. Đảo C = {(1,2)} — 4 ô kề (0,2),(2,2),(1,1),(1,3) đều là nước ⇒ **đóng**.

Kết luận: 1 đảo đóng.

**Quan sát then chốt:** trong một đảo, mọi ô đất kề nhau đều *cùng* đảo, nên ô kề của một ô đất chỉ có thể là (a) đất cùng đảo, (b) nước, hoặc (c) "ngoài lưới". Đảo bị nước bao kín ⇔ **không có ô nào kề với ngoài lưới** ⇔ **không có ô nào nằm trên biên**.

In [8]:
def label_islands(grid):
    """Gán nhãn A, B, C... cho từng đảo để nhìn trực quan."""
    m, n = len(grid), len(grid[0])
    lab = [['#' if v else '?' for v in row] for row in grid]
    name = 0
    info = []
    for r in range(m):
        for c in range(n):
            if lab[r][c] == '?':
                ch = chr(ord('A') + name); name += 1
                comp = bfs_component(grid, r, c)
                for rr, cc in comp:
                    lab[rr][cc] = ch
                touches = any(on_border(rr, cc, m, n) for rr, cc in comp)
                info.append((ch, len(comp), 'chạm biên' if touches else 'ĐÓNG'))
    for row in lab:
        print(' '.join(row))
    for x in info:
        print(x)

label_islands(EX2)
print()
label_islands(EX1)

A A # B B
A # C # B
A # # # B
('A', 4, 'chạm biên')
('B', 4, 'chạm biên')
('C', 1, 'ĐÓNG')

# # # # # # # A
# B B B B # # A
# B # B # # # A
# B B B B # C #
# # # # # # # D
('A', 3, 'chạm biên')
('B', 10, 'ĐÓNG')
('C', 1, 'ĐÓNG')
('D', 1, 'chạm biên')


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng ngây thơ:** với **mỗi** ô đất, chạy một BFS mới (không dùng chung `visited`) để lấy toàn bộ đảo chứa nó, rồi kiểm tra đảo có chạm biên không. Để không đếm một đảo nhiều lần, chỉ đếm khi ô hiện tại là **ô nhỏ nhất** (theo thứ tự `(r, c)`) của đảo — gọi là "đại diện".

Cách này đúng nhưng lãng phí: một đảo `k` ô bị duyệt lại `k` lần.

In [9]:
def closed_brute(grid):
    m, n = len(grid), len(grid[0])
    count = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] != 0:
                continue
            comp = bfs_component(grid, r, c)          # BFS mới mỗi lần
            if min(comp) != (r, c):                  # không phải đại diện
                continue
            if not any(on_border(rr, cc, m, n) for rr, cc in comp):
                count += 1
    return count

for g, want in examples:
    print(closed_brute(g), 'mong đợi', want)

2 mong đợi 2
1 mong đợi 1
2 mong đợi 2


**Độ phức tạp:** gọi `N = m·n` (≤ 10⁴). Mỗi ô đất chạy BFS tốn tới `O(N)` ⇒ **`O(N²)` thời gian** (≈ 10⁸ ở trường hợp toàn đất — quá chậm), `O(N)` bộ nhớ.

**Nút thắt:** cùng một đảo bị duyệt lặp đi lặp lại. Nếu đã duyệt một đảo rồi thì mọi ô của nó đã biết kết quả — chỉ cần một tập `visited` **dùng chung** cho cả lưới.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — điều kiện đóng:** *đảo đóng ⇔ không ô nào của đảo nằm trên biên.* (Chứng minh ở mục 10.)

**Insight 2 — visited dùng chung:** mỗi ô đất thuộc đúng một đảo, nên duyệt mỗi ô **đúng một lần** là đủ ⇒ tổng `O(m·n)`.

**Invariant khi quét:** sau khi xử lý xong ô `(r, c)` theo thứ tự quét, mọi đảo có ô bắt đầu ≤ `(r, c)` đã được duyệt **trọn vẹn** và đã được phân loại đóng/mở đúng.

**Các quy tắc "nếu… thì…":**
- Nếu gặp ô đất chưa thăm ⇒ đó là đảo mới ⇒ duyệt hết đảo, ghi nhận cờ `touch_border`.
- Nếu trong lúc duyệt gặp ô biên ⇒ bật cờ `touch_border = True` nhưng **vẫn duyệt tiếp** để đánh dấu hết đảo.
- Nếu đảo duyệt xong mà cờ vẫn `False` ⇒ cộng 1.
- Biến tấu: nếu trước hết "nhấn chìm" (lấp) mọi đảo chạm biên ⇒ mọi đảo còn lại đều đóng ⇒ bài toán trở thành **đếm đảo** thuần tuý.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS mỗi đảo một lần + cờ chạm biên

Từng bước:
1. `seen` là ma trận boolean `m × n`.
2. Quét mọi ô; gặp ô đất chưa thăm ⇒ BFS từ đó.
3. Trong BFS, mỗi ô lấy ra: nếu nằm trên biên ⇒ `closed = False`. Đẩy các ô kề là đất chưa thăm.
4. Hết BFS ⇒ nếu `closed` vẫn đúng ⇒ `ans += 1`.

In [10]:
def closed_bfs(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    ans = 0
    for sr in range(m):
        for sc in range(n):
            if grid[sr][sc] != 0 or seen[sr][sc]:
                continue
            closed = True
            seen[sr][sc] = True
            q = deque([(sr, sc)])
            while q:
                r, c = q.popleft()
                if r == 0 or r == m - 1 or c == 0 or c == n - 1:
                    closed = False          # bật cờ nhưng KHÔNG dừng
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == 0 and not seen[nr][nc]:
                        seen[nr][nc] = True
                        q.append((nr, nc))
            if closed:
                ans += 1
    return ans

for g, want in examples:
    print(closed_bfs(g), 'mong đợi', want)

2 mong đợi 2
1 mong đợi 1
2 mong đợi 2


Bản nộp LeetCode của Cách 2 (không sửa input):

In [11]:
class SolutionBFS:
    def closedIsland(self, grid: list[list[int]]) -> int:
        m, n = len(grid), len(grid[0])
        seen = [[False] * n for _ in range(m)]
        ans = 0
        for sr in range(m):
            for sc in range(n):
                if grid[sr][sc] or seen[sr][sc]:
                    continue
                closed = True
                seen[sr][sc] = True
                q = deque([(sr, sc)])
                while q:
                    r, c = q.popleft()
                    if r in (0, m - 1) or c in (0, n - 1):
                        closed = False
                    for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                        if 0 <= nr < m and 0 <= nc < n and not grid[nr][nc] and not seen[nr][nc]:
                            seen[nr][nc] = True
                            q.append((nr, nc))
                ans += closed
        return ans

print([SolutionBFS().closedIsland(g) for g, _ in examples])

[2, 1, 2]


**Độ phức tạp Cách 2:** mỗi ô vào hàng đợi tối đa 1 lần, mỗi lần xét 4 ô kề ⇒ **`O(m·n)` thời gian**, **`O(m·n)` bộ nhớ** (ma trận `seen` + hàng đợi).

### 7.2 Cách 3 — Lấp biên trước, rồi đếm đảo

Ý tưởng "hai pha":
1. **Pha 1:** duyệt các ô trên 4 cạnh; ô nào là đất thì flood-fill (biến cả đảo thành nước `1`). Sau pha này, không còn đảo nào chạm biên.
2. **Pha 2:** đếm số đảo còn lại như bài 200 — **mọi** đảo còn lại đều đóng.

Ưu điểm: logic tách bạch, không cần cờ. Nhược: sửa lưới ⇒ ta làm trên bản sao.

In [12]:
def closed_border_fill(grid):
    g = [row[:] for row in grid]          # bản sao để không sửa input
    m, n = len(g), len(g[0])

    def sink(sr, sc):                      # flood-fill bằng stack tường minh
        if g[sr][sc] != 0:
            return
        g[sr][sc] = 1
        stack = [(sr, sc)]
        while stack:
            r, c = stack.pop()
            for nr, nc in neighbors(r, c, m, n):
                if g[nr][nc] == 0:
                    g[nr][nc] = 1
                    stack.append((nr, nc))

    # Pha 1: nhấn chìm các đảo chạm biên
    for r in range(m):
        sink(r, 0); sink(r, n - 1)
    for c in range(n):
        sink(0, c); sink(m - 1, c)
    # Pha 2: đếm đảo còn lại
    ans = 0
    for r in range(m):
        for c in range(n):
            if g[r][c] == 0:
                ans += 1
                sink(r, c)
    return ans

for g, want in examples:
    print(closed_border_fill(g), 'mong đợi', want)

2 mong đợi 2
1 mong đợi 1
2 mong đợi 2


Xem lưới EX1 sau Pha 1 — mọi ô đất còn lại đều thuộc đảo đóng:

In [13]:
def after_phase1(grid):
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    stack = [(r, c) for r in range(m) for c in range(n)
             if (r in (0, m - 1) or c in (0, n - 1)) and g[r][c] == 0]
    for r, c in stack:
        g[r][c] = 1
    while stack:
        r, c = stack.pop()
        for nr, nc in neighbors(r, c, m, n):
            if g[nr][nc] == 0:
                g[nr][nc] = 1
                stack.append((nr, nc))
    return g

print('Trước:'); show(EX1)
print('Sau pha 1:'); show(after_phase1(EX1))

Trước:
# # # # # # # .
# . . . . # # .
# . # . # # # .
# . . . . # . #
# # # # # # # .
Sau pha 1:
# # # # # # # #
# . . . . # # #
# . # . # # # #
# . . . . # . #
# # # # # # # #


Bản nộp LeetCode của Cách 3 (sửa trực tiếp `grid` — LeetCode chấp nhận, nhưng hãy biết là đang sửa input):

In [14]:
class SolutionBorderFill:
    def closedIsland(self, grid: list[list[int]]) -> int:
        m, n = len(grid), len(grid[0])

        def sink(r, c):
            stack = [(r, c)]
            grid[r][c] = 1
            while stack:
                r, c = stack.pop()
                for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                    if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == 0:
                        grid[nr][nc] = 1
                        stack.append((nr, nc))

        for r in range(m):
            for c in range(n):
                if (r in (0, m - 1) or c in (0, n - 1)) and grid[r][c] == 0:
                    sink(r, c)
        ans = 0
        for r in range(1, m - 1):
            for c in range(1, n - 1):
                if grid[r][c] == 0:
                    ans += 1
                    sink(r, c)
        return ans

print([SolutionBorderFill().closedIsland([row[:] for row in g]) for g, _ in examples])

[2, 1, 2]


**Độ phức tạp Cách 3:** mỗi ô bị lấp tối đa 1 lần ⇒ **`O(m·n)` thời gian**; bộ nhớ phụ là stack `O(m·n)` (trường hợp xấu) — nếu sửa trực tiếp input thì không cần ma trận `seen`.

### 7.3 Cách 4 — DFS trả về `closed` (đệ quy)

Định nghĩa `dfs(r, c)` trả về `True` nếu phần đảo đi qua `(r, c)` **không** thoát ra biên:
- Nếu `(r, c)` **ra ngoài lưới** ⇒ đã "thoát" ⇒ trả `False`.
- Nếu là nước hoặc đã thăm ⇒ không ảnh hưởng ⇒ trả `True`.
- Ngược lại đánh dấu, gọi cả 4 hướng và trả về **AND** của 4 kết quả.

⚠️ Bẫy kinh điển: viết `return dfs(up) and dfs(down) and ...` sẽ **đoản mạch** (short-circuit) — khi `dfs(up)` trả `False`, ba lời gọi còn lại **không chạy**, phần đảo đó không được đánh dấu ⇒ về sau bị đếm thành đảo riêng. Phải gọi hết rồi mới AND (xem Lỗi 1 ở mục 13).

In [15]:
def closed_dfs_flag(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(max(old, 4 * m * n + 100))

    def dfs(r, c):
        if r < 0 or r >= m or c < 0 or c >= n:
            return False                    # thoát ra ngoài lưới
        if grid[r][c] == 1 or seen[r][c]:
            return True
        seen[r][c] = True
        results = [dfs(r + dr, c + dc) for dr, dc in DIRS]   # gọi HẾT 4 hướng
        return all(results)

    ans = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] == 0 and not seen[r][c] and dfs(r, c):
                ans += 1
    sys.setrecursionlimit(old)
    return ans

for g, want in examples:
    print(closed_dfs_flag(g), 'mong đợi', want)

2 mong đợi 2
1 mong đợi 1
2 mong đợi 2


Mẹo gọn khác: dùng toán tử `&` trên `bool` (không đoản mạch) — `return dfs(a) & dfs(b) & dfs(c) & dfs(d)`. Hoặc tích luỹ: `ok = True; for ...: ok = dfs(...) and ok` (đặt lời gọi **bên trái** `and`).

Phiên bản **stack tường minh** của cùng ý tưởng (an toàn với lưới 100×100 hình con rắn):

In [16]:
def closed_dfs_stack(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    ans = 0
    for sr in range(m):
        for sc in range(n):
            if grid[sr][sc] or seen[sr][sc]:
                continue
            seen[sr][sc] = True
            stack = [(sr, sc)]
            closed = True
            while stack:
                r, c = stack.pop()
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if not (0 <= nr < m and 0 <= nc < n):
                        closed = False          # có hướng đi ra ngoài lưới
                    elif grid[nr][nc] == 0 and not seen[nr][nc]:
                        seen[nr][nc] = True
                        stack.append((nr, nc))
            ans += closed
    return ans

for g, want in examples:
    print(closed_dfs_stack(g), 'mong đợi', want)

2 mong đợi 2
1 mong đợi 1
2 mong đợi 2


**Độ phức tạp Cách 4:** `O(m·n)` thời gian; bộ nhớ `O(m·n)` cho `seen` + độ sâu đệ quy/stack tới `O(m·n)`.

### 7.4 Cách 5 — Union-Find với đỉnh ảo "BIÊN"

1. Đánh số ô `(r, c)` thành `r*n + c`; thêm một đỉnh ảo `BORDER = m*n`.
2. Với mỗi ô đất: nếu ở biên ⇒ `union(ô, BORDER)`; với ô đất bên phải và bên dưới ⇒ `union` (chỉ cần 2 hướng vì quét hết lưới).
3. Đáp án = số **gốc khác nhau** của các ô đất, **trừ** nhóm chứa `BORDER`.

Ưu điểm: không cần duyệt đồ thị; hợp với biến thể "lưới thay đổi dần" (thêm đất online).

In [17]:
def closed_union_find(grid):
    m, n = len(grid), len(grid[0])
    BORDER = m * n
    parent = list(range(m * n + 1))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    for r in range(m):
        for c in range(n):
            if grid[r][c]:
                continue
            idx = r * n + c
            if r in (0, m - 1) or c in (0, n - 1):
                union(idx, BORDER)
            if r + 1 < m and grid[r + 1][c] == 0:
                union(idx, idx + n)
            if c + 1 < n and grid[r][c + 1] == 0:
                union(idx, idx + 1)
    border_root = find(BORDER)
    roots = {find(r * n + c) for r in range(m) for c in range(n) if grid[r][c] == 0}
    roots.discard(border_root)
    return len(roots)

for g, want in examples:
    print(closed_union_find(g), 'mong đợi', want)

2 mong đợi 2
1 mong đợi 1
2 mong đợi 2


Bản nộp LeetCode của Cách 5:

In [18]:
class SolutionUF:
    def closedIsland(self, grid: list[list[int]]) -> int:
        m, n = len(grid), len(grid[0])
        parent = list(range(m * n + 1))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        for r in range(m):
            for c in range(n):
                if grid[r][c]:
                    continue
                i = r * n + c
                pairs = []
                if r in (0, m - 1) or c in (0, n - 1):
                    pairs.append(m * n)
                if r + 1 < m and not grid[r + 1][c]:
                    pairs.append(i + n)
                if c + 1 < n and not grid[r][c + 1]:
                    pairs.append(i + 1)
                for j in pairs:
                    a, b = find(i), find(j)
                    if a != b:
                        parent[a] = b
        roots = {find(r * n + c) for r in range(m) for c in range(n) if not grid[r][c]}
        roots.discard(find(m * n))
        return len(roots)

print([SolutionUF().closedIsland(g) for g, _ in examples])

[2, 1, 2]


**Độ phức tạp Cách 5:** `O(m·n·α(m·n))` ≈ `O(m·n)` thời gian (α là hàm Ackermann ngược, ≤ 4 trong thực tế), `O(m·n)` bộ nhớ cho mảng `parent`.

## 8. TRACE từng bước

Trace **Cách 2 (BFS + cờ)** trên Ví dụ 2. Bảng tóm tắt các đảo được phát hiện theo thứ tự quét:

| Ô bắt đầu | Các ô lấy ra khỏi hàng đợi | Có ô biên? | `closed` | `ans` sau đảo |
|---|---|---|---|---|
| (0,0) | (0,0),(1,0),(0,1),(2,0) | có (0,0) | False | 0 |
| (0,3) | (0,3),(0,4),(1,4),(2,4) | có (0,3) | False | 0 |
| (1,2) | (1,2) | không | True | 1 |

Hàm `trace_closed_bfs` in chi tiết hàng đợi qua từng bước:

In [19]:
def trace_closed_bfs(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    ans = 0
    for sr in range(m):
        for sc in range(n):
            if grid[sr][sc] or seen[sr][sc]:
                continue
            print(f'== Đảo mới bắt đầu tại ({sr},{sc})')
            closed = True
            seen[sr][sc] = True
            q = deque([(sr, sc)])
            while q:
                r, c = q.popleft()
                border = r in (0, m - 1) or c in (0, n - 1)
                if border:
                    closed = False
                pushed = []
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == 0 and not seen[nr][nc]:
                        seen[nr][nc] = True
                        q.append((nr, nc))
                        pushed.append((nr, nc))
                print(f'   lấy ({r},{c}) biên={border!s:5} đẩy={pushed} hàng đợi={list(q)} closed={closed}')
            ans += closed
            print(f'   => closed={closed}, ans={ans}')
    return ans

print('Kết quả:', trace_closed_bfs(EX2))

== Đảo mới bắt đầu tại (0,0)
   lấy (0,0) biên=True  đẩy=[(1, 0), (0, 1)] hàng đợi=[(1, 0), (0, 1)] closed=False
   lấy (1,0) biên=True  đẩy=[(2, 0)] hàng đợi=[(0, 1), (2, 0)] closed=False
   lấy (0,1) biên=True  đẩy=[] hàng đợi=[(2, 0)] closed=False
   lấy (2,0) biên=True  đẩy=[] hàng đợi=[] closed=False
   => closed=False, ans=0
== Đảo mới bắt đầu tại (0,3)
   lấy (0,3) biên=True  đẩy=[(0, 4)] hàng đợi=[(0, 4)] closed=False
   lấy (0,4) biên=True  đẩy=[(1, 4)] hàng đợi=[(1, 4)] closed=False
   lấy (1,4) biên=True  đẩy=[(2, 4)] hàng đợi=[(2, 4)] closed=False
   lấy (2,4) biên=True  đẩy=[] hàng đợi=[] closed=False
   => closed=False, ans=0
== Đảo mới bắt đầu tại (1,2)
   lấy (1,2) biên=False đẩy=[] hàng đợi=[] closed=True
   => closed=True, ans=1
Kết quả: 1


Trace **Cách 4 (DFS trả cờ)** trên một lưới nhỏ — in lời gọi theo độ sâu để thấy rõ giá trị trả về được AND lên như thế nào:

In [20]:
def trace_dfs_flag(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]

    def dfs(r, c, depth):
        pad = '  ' * depth
        if not (0 <= r < m and 0 <= c < n):
            print(f'{pad}({r},{c}) ngoài lưới -> False')
            return False
        if grid[r][c] == 1 or seen[r][c]:
            return True
        seen[r][c] = True
        print(f'{pad}vào ({r},{c})')
        res = [dfs(r + dr, c + dc, depth + 1) for dr, dc in DIRS]
        print(f'{pad}rời ({r},{c}) kết quả 4 hướng={res} -> {all(res)}')
        return all(res)

    ans = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] == 0 and not seen[r][c]:
                ok = dfs(r, c, 0)
                ans += ok
                print(f'== đảo tại ({r},{c}) closed={ok}, ans={ans}\n')
    return ans

small = [[1, 1, 1, 1],
         [1, 0, 0, 1],
         [1, 1, 0, 0]]
show(small)
print('Kết quả:', trace_dfs_flag(small))

# # # #
# . . #
# # . .
vào (1,1)
  vào (1,2)
    vào (2,2)
      (3,2) ngoài lưới -> False
      vào (2,3)
        (3,3) ngoài lưới -> False
        (2,4) ngoài lưới -> False
      rời (2,3) kết quả 4 hướng=[False, True, False, True] -> False
    rời (2,2) kết quả 4 hướng=[False, True, False, True] -> False
  rời (1,2) kết quả 4 hướng=[False, True, True, True] -> False
rời (1,1) kết quả 4 hướng=[True, True, False, True] -> False
== đảo tại (1,1) closed=False, ans=0

Kết quả: 0


## 9. Code nộp LeetCode

Chọn Cách 2 (BFS + cờ chạm biên): không đệ quy, không sửa input, dễ đọc.

In [21]:
from collections import deque

class Solution:
    def closedIsland(self, grid: list[list[int]]) -> int:
        m, n = len(grid), len(grid[0])
        seen = [[False] * n for _ in range(m)]
        ans = 0
        for sr in range(m):
            for sc in range(n):
                if grid[sr][sc] == 1 or seen[sr][sc]:
                    continue
                closed = True
                seen[sr][sc] = True
                q = deque([(sr, sc)])
                while q:
                    r, c = q.popleft()
                    if r == 0 or r == m - 1 or c == 0 or c == n - 1:
                        closed = False
                    for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                        if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == 0 and not seen[nr][nc]:
                            seen[nr][nc] = True
                            q.append((nr, nc))
                if closed:
                    ans += 1
        return ans

for g, want in examples:
    got = Solution().closedIsland(g)
    print(got, want, 'OK' if got == want else 'SAI')

2 2 OK
1 1 OK
2 2 OK


## 10. Vì sao thuật toán đúng

**Bổ đề (điều kiện đóng).** Một đảo `I` là đóng ⇔ không ô nào của `I` nằm trên biên lưới.

- (⇐) Giả sử không ô nào của `I` ở biên. Xét một ô `x ∈ I` và một ô kề `y` (theo 4 hướng). Vì `x` không ở biên nên `y` nằm trong lưới. Nếu `y` là đất thì `y` liên thông với `x` nên `y ∈ I` (vì đảo là nhóm **cực đại**). Vậy mọi ô kề của đảo nằm ngoài `I` đều là nước ⇒ đảo bị nước bao kín bốn phía ⇒ đóng.
- (⇒) Nếu có ô `x ∈ I` nằm trên biên, chẳng hạn ở cột 0, thì phía trái của `x` là "ngoài lưới", không phải nước ⇒ đảo không bị bao kín ⇒ không đóng.

**Tính đúng của Cách 2.** Mỗi lần gặp ô đất chưa thăm, BFS thăm **đúng** tập ô của đảo chứa nó (BFS thăm mọi ô đến được qua các ô đất, và chỉ những ô đó). Vì `seen` dùng chung, mỗi đảo được bắt đầu đúng một lần. Cờ `closed` = "không ô nào được lấy ra nằm trên biên" — đúng bằng điều kiện của bổ đề vì BFS lấy ra **mọi** ô của đảo (ta không dừng sớm). Do đó `ans` = số đảo đóng.

**Cách 3.** Sau Pha 1, mọi đảo chạm biên bị xoá hết (flood-fill từ ô biên lấp cả đảo), còn các đảo không chạm biên giữ nguyên ⇒ số đảo còn lại = số đảo đóng.

**Cách 5.** Hai ô đất cùng gốc ⇔ cùng đảo, hoặc cả hai cùng nối với `BORDER`. Mọi đảo chạm biên dồn vào nhóm của `BORDER`; mọi đảo đóng là một nhóm riêng ⇒ đếm gốc trừ nhóm `BORDER`.

## 11. Phân tích Time / Space

Gọi `m` = số hàng, `n` = số cột, `N = m·n ≤ 10⁴`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(N²)` | `O(N)` | BFS lại cho từng ô đất |
| 2. BFS + cờ | `O(N)` | `O(N)` | mỗi ô vào hàng đợi ≤ 1 lần |
| 3. Lấp biên | `O(N)` | `O(N)` (stack), `O(1)` nếu sửa input & đệ quy không tính | sửa lưới |
| 4. DFS trả cờ | `O(N)` | `O(N)` + độ sâu đệ quy `O(N)` | cần tăng recursion limit |
| 5. Union-Find | `O(N·α(N))` | `O(N)` | hợp cho bài online |

Không có khác biệt average/worst đáng kể với Cách 2–5: luôn duyệt toàn bộ lưới một lần. Worst case về **bộ nhớ hàng đợi/stack** là lưới toàn đất (`O(N)` phần tử).

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [22]:
ALL = {
    'bfs': closed_bfs,
    'border_fill': closed_border_fill,
    'dfs_flag': closed_dfs_flag,
    'dfs_stack': closed_dfs_stack,
    'union_find': closed_union_find,
    'submit': lambda g: Solution().closedIsland(g),
}

edge = [
    ([[0]], 0),                                   # 1 ô đất nằm ở biên
    ([[1]], 0),                                   # 1 ô nước
    ([[1, 1, 1], [1, 0, 1], [1, 1, 1]], 1),       # 1 ô đất ở giữa
    ([[0, 0, 0], [0, 0, 0], [0, 0, 0]], 0),       # toàn đất -> chạm biên
    ([[1, 1, 1, 1], [1, 0, 0, 1], [1, 0, 0, 1], [1, 1, 1, 1]], 1),
    ([[1, 1, 1, 1, 1], [1, 0, 1, 0, 1], [1, 1, 1, 1, 1]], 2),
    ([[1, 0, 1]], 0),                             # 1 hàng: mọi ô là biên
    ([[1], [0], [1]], 0),                         # 1 cột
    ([[1, 1, 1, 1], [1, 0, 0, 1], [1, 1, 0, 1], [1, 1, 0, 1]], 0),  # đảo "rò" xuống biên dưới
]
for name, f in ALL.items():
    for g, want in examples + edge:
        snapshot = [row[:] for row in g]
        got = f(g)
        assert got == want, (name, g, got, want)
        assert g == snapshot, (name, 'đã sửa input!')
print('Tất cả', len(examples + edge), 'test x', len(ALL), 'cách đều đúng và không sửa input')

Tất cả 12 test x 6 cách đều đúng và không sửa input


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Sinh lưới ngẫu nhiên kích thước 1..12, xác suất đất thay đổi để có cả lưới thưa lẫn dày.

In [23]:
rng = random.Random(1254)
for t in range(400):
    m, n = rng.randint(1, 12), rng.randint(1, 12)
    p = rng.choice([0.3, 0.5, 0.7])
    g = [[0 if rng.random() < p else 1 for _ in range(n)] for _ in range(m)]
    want = closed_brute(g)
    for name, f in ALL.items():
        got = f(g)
        assert got == want, (name, g, got, want)
print('400 lưới ngẫu nhiên: mọi cách khớp brute force')

400 lưới ngẫu nhiên: mọi cách khớp brute force


### 12.3 Test giới hạn 100×100

Ba kiểu lưới khó: toàn đất, bàn cờ (nhiều đảo 1 ô), và "con rắn" dài (độ sâu DFS lớn).

In [24]:
def snake(m, n):
    g = [[1] * n for _ in range(m)]
    for r in range(1, m - 1, 2):
        for c in range(1, n - 1):
            g[r][c] = 0
        if r + 2 < m - 1:
            col = n - 2 if (r // 2) % 2 == 0 else 1
            g[r + 1][col] = 0
    return g

big_all_land = [[0] * 100 for _ in range(100)]
checker = [[(r + c) % 2 for c in range(100)] for r in range(100)]
big_snake = snake(100, 100)
for name, g in [('toàn đất', big_all_land), ('bàn cờ', checker), ('con rắn', big_snake)]:
    t0 = time.perf_counter()
    res = {k: f(g) for k, f in ALL.items()}
    assert len(set(res.values())) == 1, res
    print(f'{name:9}: đáp án={res["bfs"]:5}  thời gian 6 cách={time.perf_counter() - t0:.3f}s')

toàn đất : đáp án=    0  thời gian 6 cách=0.070s
bàn cờ   : đáp án= 4802  thời gian 6 cách=0.031s
con rắn  : đáp án=    1  thời gian 6 cách=0.028s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `and` đoản mạch trong DFS trả cờ

```python
return dfs(r+1,c) and dfs(r-1,c) and dfs(r,c+1) and dfs(r,c-1)
```

Khi nhánh đầu trả `False`, các nhánh sau **không được gọi** ⇒ phần còn lại của đảo chưa được đánh dấu ⇒ vòng quét bên ngoài gặp lại nó và có thể đếm nhầm thành một đảo đóng mới.

In [25]:
def closed_dfs_shortcircuit_WRONG(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    def dfs(r, c):
        if not (0 <= r < m and 0 <= c < n):
            return False
        if grid[r][c] == 1 or seen[r][c]:
            return True
        seen[r][c] = True
        return dfs(r - 1, c) and dfs(r + 1, c) and dfs(r, c - 1) and dfs(r, c + 1)
    ans = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] == 0 and not seen[r][c] and dfs(r, c):
                ans += 1
    return ans

bad = [[1, 0, 1, 1],
       [1, 0, 0, 1],
       [1, 1, 1, 1]]
show(bad)
print('Sai :', closed_dfs_shortcircuit_WRONG(bad), '| Đúng:', closed_dfs_flag(bad))

# . # #
# . . #
# # # #
Sai : 1 | Đúng: 0


Giải thích: từ (0,1) gọi `dfs(-1,1)` ra ngoài ⇒ `False` ⇒ dừng luôn, ô (1,1),(1,2) chưa thăm. Vòng quét gặp (1,1): lúc này (0,1) đã `seen` nên bị coi như "nước" ⇒ trả `True` ⇒ đếm sai thành 1. **Sửa:** gọi đủ 4 hướng rồi mới gộp (`all([...])`, `&`, hoặc `ok = dfs(...) and ok`).

### Lỗi 2 — Dừng BFS ngay khi chạm biên

"Chạm biên rồi thì không đóng, `break` cho nhanh" — nhưng phần đảo chưa duyệt chưa được đánh dấu, lần quét sau sẽ bắt đầu một "đảo mới" từ phần dư đó.

In [26]:
def closed_bfs_early_break_WRONG(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    ans = 0
    for sr in range(m):
        for sc in range(n):
            if grid[sr][sc] or seen[sr][sc]:
                continue
            closed = True
            seen[sr][sc] = True
            q = deque([(sr, sc)])
            while q:
                r, c = q.popleft()
                if r in (0, m - 1) or c in (0, n - 1):
                    closed = False
                    break                          # SAI: bỏ dở đảo
                for nr, nc in neighbors(r, c, m, n):
                    if grid[nr][nc] == 0 and not seen[nr][nc]:
                        seen[nr][nc] = True
                        q.append((nr, nc))
            ans += closed
    return ans

bad3 = [[1, 0, 1, 1, 1],
        [1, 0, 0, 0, 1],
        [1, 1, 1, 0, 1],
        [1, 1, 1, 1, 1]]
show(bad3)
print('Sai :', closed_bfs_early_break_WRONG(bad3), '| Đúng:', closed_bfs(bad3))

# . # # #
# . . . #
# # # . #
# # # # #
Sai : 1 | Đúng: 0


Giải thích: BFS bắt đầu ở (0,1) — ô biên — nên lấy ra là `break` ngay, các ô (1,1),(1,2),(1,3),(2,3) chưa được đánh dấu. Vòng quét gặp (1,1) và mở một "đảo mới" không chứa ô biên nào ⇒ đếm nhầm 1. Lỗi này không hiếm: tìm ngẫu nhiên thêm một phản ví dụ khác.

In [27]:
# Tìm phản ví dụ tự động để chắc chắn lỗi là có thật
rng = random.Random(7)
for _ in range(2000):
    m, n = rng.randint(3, 6), rng.randint(3, 6)
    g = [[rng.randint(0, 1) for _ in range(n)] for _ in range(m)]
    if closed_bfs_early_break_WRONG(g) != closed_bfs(g):
        print('Phản ví dụ ngẫu nhiên:'); show(g)
        print('Sai :', closed_bfs_early_break_WRONG(g), '| Đúng:', closed_bfs(g))
        break

Phản ví dụ ngẫu nhiên:
# # # . .
# # . . #
# # # # .
# # . . #
. . # . .
Sai : 1 | Đúng: 0


**Sửa:** chỉ bật cờ `closed = False`, **không** `break`; luôn duyệt trọn đảo.

### Lỗi 3 — Đếm đảo theo kiểu bài 200 (quên điều kiện "đóng") hoặc nhầm `0`/`1`

Bê nguyên code bài 200 sẽ đếm **mọi** đảo (kể cả chạm biên), hoặc đếm nhầm các vùng nước `1`.

In [28]:
def count_all_islands_WRONG(grid):
    """Đếm mọi nhóm 0 — quên kiểm tra biên."""
    m, n = len(grid), len(grid[0])
    seen = set(); ans = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] == 0 and (r, c) not in seen:
                ans += 1
                dfs_mark(grid, r, c, seen)
    return ans

print('EX1 — sai:', count_all_islands_WRONG(EX1), '| đúng:', closed_bfs(EX1))
print('EX2 — sai:', count_all_islands_WRONG(EX2), '| đúng:', closed_bfs(EX2))

EX1 — sai: 4 | đúng: 2
EX2 — sai: 3 | đúng: 1


### Lỗi 4 — Coi "ngoài lưới" là nước

Nếu hàm DFS trả `True` khi ra ngoài lưới (tức xem ngoài lưới như nước), mọi đảo đều thành "đóng". Một biến thể khác của lỗi này: kiểm tra biên bằng `grid[r-1][c]` mà không xét `r-1 >= 0` — chỉ số `-1` lặng lẽ trỏ tới hàng cuối.

In [29]:
def closed_outside_is_water_WRONG(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    def dfs(r, c):
        if not (0 <= r < m and 0 <= c < n):
            return True                      # SAI: ngoài lưới không phải nước
        if grid[r][c] == 1 or seen[r][c]:
            return True
        seen[r][c] = True
        return all([dfs(r + dr, c + dc) for dr, dc in DIRS])
    return sum(1 for r in range(m) for c in range(n)
               if grid[r][c] == 0 and not seen[r][c] and dfs(r, c))

print('EX2 — sai:', closed_outside_is_water_WRONG(EX2), '| đúng:', closed_bfs(EX2))
print('[[0]] — sai:', closed_outside_is_water_WRONG([[0]]), '| đúng:', closed_bfs([[0]]))

EX2 — sai: 3 | đúng: 1
[[0]] — sai: 1 | đúng: 0


### Lỗi 5 — Tràn đệ quy với lưới lớn

DFS đệ quy trên đảo "con rắn" 100×100 cần độ sâu vài nghìn > giới hạn mặc định 1000. Ta minh hoạ an toàn bằng cách bắt `RecursionError` (để notebook không dừng), rồi so với bản stack.

In [30]:
def closed_dfs_default_limit(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    def dfs(r, c):
        if not (0 <= r < m and 0 <= c < n):
            return False
        if grid[r][c] == 1 or seen[r][c]:
            return True
        seen[r][c] = True
        return all([dfs(r + dr, c + dc) for dr, dc in DIRS])
    return sum(1 for r in range(m) for c in range(n)
               if grid[r][c] == 0 and not seen[r][c] and dfs(r, c))

old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    print('Đệ quy mặc định:', closed_dfs_default_limit(big_snake))
except RecursionError as e:
    print('Đệ quy mặc định: RecursionError ->', str(e)[:45])
finally:
    sys.setrecursionlimit(old)
print('Bản stack       :', closed_dfs_stack(big_snake))

Đệ quy mặc định: RecursionError -> maximum recursion depth exceeded
Bản stack       : 1


**Sửa:** dùng BFS/stack tường minh, hoặc `sys.setrecursionlimit(10**5)` ở đầu lời giải (vẫn có rủi ro tràn stack C trên một số môi trường).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Dấu hiệu nhận dạng:** "lưới 0/1", "4-directionally connected", "đếm vùng / đảo" ⇒ flood-fill (DFS/BFS) hoặc Union-Find.
- **Có chữ "surrounded" / "enclosed" / "không chạm biên"** ⇒ nghĩ ngay tới **lấp biên trước** (bài 130, 1020, 1254) — đổi bài khó thành bài đếm đảo thường.
- Viết sẵn `for nr, nc in ((r+1,c),(r-1,c),(r,c+1),(r,c-1))` + kiểm tra biên trong một dòng — giảm lỗi gõ.
- Đánh dấu **khi đẩy vào** hàng đợi/stack, không phải khi lấy ra.
- Nếu đề cho phép sửa input, ghi đè `grid[r][c] = 1` thay cho `seen` để tiết kiệm bộ nhớ và code.
- Muốn DFS đệ quy trả cờ: **gọi hết rồi mới AND**.
- Với bài này có thể bỏ qua hàng/cột biên ở pha đếm (`range(1, m-1)`) sau khi đã lấp biên.

## 15. Biến thể của bài

1. **LeetCode 200 — Number of Islands:** đếm mọi đảo (với `1` là đất).
2. **LeetCode 1020 — Number of Enclaves:** đếm **số ô** đất không thể đi ra biên — lấp biên rồi đếm ô `1` còn lại.
3. **LeetCode 130 — Surrounded Regions:** lật các vùng `'O'` bị bao kín thành `'X'` — đúng y chiến lược lấp biên (đánh dấu vùng thoát được, lật phần còn lại).
4. **LeetCode 695 — Max Area of Island:** trả về diện tích đảo lớn nhất.
5. **Biến thể tự chế:** diện tích đảo đóng lớn nhất; đếm đảo đóng khi cho phép kề 8 hướng; trả về danh sách toạ độ từng đảo đóng.

Code minh hoạ biến thể 1020 (đếm ô đất bị bao kín, ở đây dùng quy ước `0` là đất của bài 1254) và "diện tích đảo đóng lớn nhất":

In [31]:
def closed_cells_and_max_area(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    total_cells, max_area = 0, 0
    for sr in range(m):
        for sc in range(n):
            if grid[sr][sc] or seen[sr][sc]:
                continue
            seen[sr][sc] = True
            q = deque([(sr, sc)]); closed = True; size = 0
            while q:
                r, c = q.popleft(); size += 1
                if r in (0, m - 1) or c in (0, n - 1):
                    closed = False
                for nr, nc in neighbors(r, c, m, n):
                    if grid[nr][nc] == 0 and not seen[nr][nc]:
                        seen[nr][nc] = True; q.append((nr, nc))
            if closed:
                total_cells += size
                max_area = max(max_area, size)
    return total_cells, max_area

for g, _ in examples:
    print('(tổng ô đất đóng, diện tích đảo đóng lớn nhất) =', closed_cells_and_max_area(g))

(tổng ô đất đóng, diện tích đảo đóng lớn nhất) = (11, 10)
(tổng ô đất đóng, diện tích đảo đóng lớn nhất) = (1, 1)
(tổng ô đất đóng, diện tích đảo đóng lớn nhất) = (17, 16)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | BFS lại cho từng ô đất, đếm theo ô đại diện | `O(N²)` | `O(N)` | Chỉ để kiểm chứng / lưới rất nhỏ |
| 2. BFS + cờ | Duyệt mỗi đảo 1 lần, bật cờ nếu gặp ô biên | `O(N)` | `O(N)` | Mặc định khi đi phỏng vấn: rõ ràng, không đệ quy |
| 3. Lấp biên trước | Xoá đảo chạm biên rồi đếm đảo | `O(N)` | `O(N)` stack (có thể sửa input) | Khi muốn tái dùng code "đếm đảo"; các bài "surrounded" |
| 4. DFS trả cờ | `dfs` trả `True` nếu không thoát ra ngoài; AND 4 hướng | `O(N)` | `O(N)` + đệ quy | Code ngắn; nhớ bẫy đoản mạch & recursion limit |
| 5. Union-Find | Gộp ô kề + đỉnh ảo BIÊN, đếm gốc | `O(N·α)` | `O(N)` | Bài có cập nhật online, hoặc khi đã quen DSU |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Không chạy code, đoán `closed_bfs` trả gì cho:
```
[[1,1,1,1,1],
 [1,0,1,0,1],
 [1,1,0,1,1],
 [1,0,1,0,1],
 [1,1,1,1,1]]
```

### Bài 2 (tính tay)
Với lưới dưới đây, hãy liệt kê các đảo (toạ độ), đảo nào chạm biên, và đáp án:
```
[[1,1,0,1],
 [1,0,0,1],
 [1,1,1,1],
 [1,0,1,1],
 [1,1,1,1]]
```

### Bài 3 (điền TODO) — Number of Enclaves theo quy ước bài này
Hoàn thiện `count_enclave_cells(grid)`: trả về **số ô đất** (`0`) không thể đi tới biên lưới.

### Bài 4 (tự viết lại)
Viết lại Cách 3 (lấp biên) bằng **BFS** thay vì stack, và **không** sửa input: `closed_island_bfs_fill(grid)`.

### Bài 5 (mini challenge) — kề 8 hướng
Nếu đảo được định nghĩa liên thông theo **8 hướng** (tính cả đường chéo), viết `closed_island_8dir(grid)`. Đảo đóng vẫn là đảo không có ô nào trên biên.

In [32]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_enclave_cells(grid):
    m, n = len(grid), len(grid[0])
    g = [row[:] for row in grid]
    # TODO 1: lấp (biến thành 1) mọi ô đất liên thông với một ô biên
    # TODO 2: đếm số ô 0 còn lại và trả về
    pass

# Bài 4 — TODO
def closed_island_bfs_fill(grid):
    # TODO: pha 1 BFS từ mọi ô đất trên biên, đánh dấu vào seen
    # TODO: pha 2 đếm số lần bắt đầu BFS mới từ ô đất chưa thăm
    pass

# Bài 5 — TODO
def closed_island_8dir(grid):
    # TODO: giống Cách 2 nhưng dùng 8 hướng
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ô giữa (2,2) có liên thông chéo với (1,1) không? (2) Theo 4 hướng thì không — mỗi ô `0` là một đảo riêng. (3) Có 5 ô đất, không ô nào ở biên ⇒ 5 đảo đóng.

**Bài 2.** (1) Tìm các nhóm `0`. (2) Nhóm {(0,2),(1,1),(1,2)} có (0,2) ở hàng 0. (3) Nhóm {(3,1)} không chạm biên ⇒ đáp án 1.

**Bài 3.** (1) Dùng lại pha 1 của Cách 3. (2) Sau khi lấp, mọi ô `0` còn lại đều là ô bị bao kín. (3) `sum(row.count(0) for row in g)`.

**Bài 4.** (1) Khởi tạo hàng đợi chứa mọi ô đất trên biên, đánh dấu `seen`. (2) BFS một lần đánh dấu mọi ô thoát được (multi-source BFS). (3) Pha 2: mỗi ô đất chưa `seen` mở BFS mới và `ans += 1`.

**Bài 5.** (1) Thay `DIRS` bằng 8 cặp `(dr, dc)` với `dr, dc ∈ {-1,0,1}` trừ `(0,0)`. (2) Điều kiện đóng không đổi: không ô nào trên biên. (3) Chú ý: kề chéo có thể nối hai đảo vốn tách rời theo 4 hướng — đáp án Bài 1 khi đó chỉ còn 1.

## 19. Đáp án tham khảo

Đáp án Bài 1 và 2 kiểm tra bằng code; Bài 3–5 có lời giải đầy đủ và đối chiếu với brute force / cách khác.

In [33]:
# Bài 1
B1 = [[1,1,1,1,1],[1,0,1,0,1],[1,1,0,1,1],[1,0,1,0,1],[1,1,1,1,1]]
assert closed_bfs(B1) == 5
# Bài 2
B2 = [[1,1,0,1],[1,0,0,1],[1,1,1,1],[1,0,1,1],[1,1,1,1]]
assert closed_bfs(B2) == 1

# Bài 3
def count_enclave_cells_sol(grid):
    m, n = len(grid), len(grid[0])
    g = [row[:] for row in grid]
    stack = [(r, c) for r in range(m) for c in range(n)
             if (r in (0, m - 1) or c in (0, n - 1)) and g[r][c] == 0]
    for r, c in stack:
        g[r][c] = 1
    while stack:
        r, c = stack.pop()
        for nr, nc in neighbors(r, c, m, n):
            if g[nr][nc] == 0:
                g[nr][nc] = 1
                stack.append((nr, nc))
    return sum(row.count(0) for row in g)

# Bài 4
def closed_island_bfs_fill_sol(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    def bfs(starts):
        q = deque(starts)
        for r, c in starts:
            seen[r][c] = True
        while q:
            r, c = q.popleft()
            for nr, nc in neighbors(r, c, m, n):
                if grid[nr][nc] == 0 and not seen[nr][nc]:
                    seen[nr][nc] = True
                    q.append((nr, nc))
    bfs([(r, c) for r in range(m) for c in range(n)
         if (r in (0, m - 1) or c in (0, n - 1)) and grid[r][c] == 0])
    ans = 0
    for r in range(m):
        for c in range(n):
            if grid[r][c] == 0 and not seen[r][c]:
                ans += 1
                bfs([(r, c)])
    return ans

# Bài 5
DIRS8 = [(dr, dc) for dr in (-1, 0, 1) for dc in (-1, 0, 1) if (dr, dc) != (0, 0)]
def closed_island_8dir_sol(grid):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    ans = 0
    for sr in range(m):
        for sc in range(n):
            if grid[sr][sc] or seen[sr][sc]:
                continue
            seen[sr][sc] = True
            q = deque([(sr, sc)]); closed = True
            while q:
                r, c = q.popleft()
                if r in (0, m - 1) or c in (0, n - 1):
                    closed = False
                for dr, dc in DIRS8:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == 0 and not seen[nr][nc]:
                        seen[nr][nc] = True; q.append((nr, nc))
            ans += closed
    return ans

assert closed_island_8dir_sol(B1) == 1
assert count_enclave_cells_sol(EX1) == closed_cells_and_max_area(EX1)[0]
rng = random.Random(2024)
for _ in range(300):
    m, n = rng.randint(1, 10), rng.randint(1, 10)
    g = [[rng.randint(0, 1) for _ in range(n)] for _ in range(m)]
    assert closed_island_bfs_fill_sol(g) == closed_brute(g)
    assert count_enclave_cells_sol(g) == closed_cells_and_max_area(g)[0]
print('Đáp án Bài 1–5: tất cả assert đều qua')

Đáp án Bài 1–5: tất cả assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm thành phần thoả điều kiện trên lưới":**

```python
seen = [[False]*n for _ in range(m)]
ans = 0
for sr in range(m):
    for sc in range(n):
        if not is_land(sr, sc) or seen[sr][sc]:
            continue
        ok = True
        seen[sr][sc] = True
        q = deque([(sr, sc)])
        while q:
            r, c = q.popleft()
            if violates(r, c):          # ví dụ: nằm trên biên
                ok = False              # KHÔNG break
            for nr, nc in ((r+1,c),(r-1,c),(r,c+1),(r,c-1)):
                if 0 <= nr < m and 0 <= nc < n and is_land(nr, nc) and not seen[nr][nc]:
                    seen[nr][nc] = True
                    q.append((nr, nc))
        ans += ok
```

**Template "lấp biên trước":** flood-fill từ mọi ô đất trên 4 cạnh → phần còn lại là vùng bị bao kín.

**Checklist trước khi submit:**
- [ ] Đúng quy ước: **`0` là đất**, `1` là nước.
- [ ] Kiểm tra `0 <= nr < m and 0 <= nc < n` **trước** khi đọc `grid[nr][nc]`.
- [ ] Đánh dấu `seen` khi **đẩy** vào hàng đợi.
- [ ] Không `break` khi chạm biên — phải duyệt trọn đảo.
- [ ] DFS trả cờ: gọi đủ 4 hướng rồi mới AND (không dùng `and` đoản mạch).
- [ ] Lưới 1 hàng / 1 cột ⇒ đáp án 0.
- [ ] Đệ quy sâu? ⇒ BFS/stack hoặc tăng recursion limit.
- [ ] Chữ ký đúng: `class Solution: def closedIsland(self, grid) -> int`.